In [0]:
import importlib
import configs.constant as constants
import utils.transformation_function as transformation_functions
importlib.reload(constants)
importlib.reload(transformation_functions)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from configs.constant import *
from pyspark.sql.window import *


df_silver = spark.read.format("delta") \
    .load(f"abfss://{SILVER_DATA}@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/clean_data/")

In [0]:
df_silver.createOrReplaceTempView("silver")

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    STORAGE_ACCOUNT_ACCESS_KEY
)

In [0]:
try:
    last_processed = spark.table(f"{CATALOG}.{SCHEMA}.fct_events") \
        .agg(max("event_timestamp")) \
        .collect()[0][0]
except:
    last_processed = None

if last_processed:
    df = df_silver.filter(col("event_timestamp") > last_processed)
else:
    df = df_silver

In [0]:

window = Window.orderBy("product_id")

dim_product = df.select("product_id", "category") \
    .dropDuplicates(["product_id"]) \
    .withColumn("product_key", dense_rank().over(window))

dim_product.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_product")

In [0]:


window = Window.orderBy("user_id")

dim_user = df.select("user_id", "region", "device_type") \
    .dropDuplicates(["user_id"]) \
    .withColumn("user_key", dense_rank().over(window))

dim_user.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.dim_user")

In [0]:
window = Window.orderBy("region")

dim_region = df.select("region") \
    .dropDuplicates(["region"]) \
    .withColumn("region_key", dense_rank().over(window))

dim_region.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.dim_region")

In [0]:
window = Window.orderBy("event_date")

dim_date = df.select(
    "event_date",
    "event_hour",
    weekofyear("event_timestamp").alias("week")
).dropDuplicates(["event_date"]) \
.withColumn("date_key", dense_rank().over(window))

dim_date.write.mode("overwrite").saveAsTable("retail_catalog.gold.dim_date")

In [0]:
from pyspark.sql.functions import col, broadcast

# aliases (IMPORTANT)
f = df.alias("f")
p = dim_product.alias("p")
u = dim_user.alias("u")
d = dim_date.alias("d")
r = dim_region.alias("r")

fct_events = f \
    .join(broadcast(p), col("f.product_id") == col("p.product_id"), "left") \
    .join(broadcast(u), col("f.user_id") == col("u.user_id"), "left") \
    .join(broadcast(d), col("f.event_date") == col("d.event_date"), "left") \
    .join(broadcast(r), col("f.region") == col("r.region"), "left") \
    .select(
        col("f.event_id"),
        col("p.product_key"),
        col("u.user_key"),
        col("d.date_key"),
        col("r.region_key"),
        col("f.category"),
        col("f.amount"),
        col("f.product_id"),
        col("f.user_id"),
        col("f.event_type"),
        col("f.event_timestamp"),
        col("f.event_date"),
        col("f.event_hour")
    ) \
    .dropDuplicates(["event_id"])   #  important

In [0]:
df.printSchema()

In [0]:
fct_events.write.format("delta") \
.mode("append") \
.saveAsTable("retail_catalog.gold.fct_events")

In [0]:
spark.sql("""
SELECT
  event_date,
  COUNT(CASE WHEN event_type='search' THEN 1 END) AS searches,
  COUNT(CASE WHEN event_type='click' THEN 1 END) AS clicks,
  COUNT(CASE WHEN event_type='cart' THEN 1 END) AS carts,
  COUNT(CASE WHEN event_type='checkout' THEN 1 END) AS checkouts,
  COUNT(CASE WHEN event_type='purchase' THEN 1 END) AS purchases
FROM retail_catalog.gold.fct_events
GROUP BY event_date
""").write.mode("overwrite").saveAsTable("retail_catalog.gold.gold_funnel_daily")

In [0]:
spark.sql("""
SELECT
  category,
  event_date,
  SUM(amount) AS total_revenue,
  COUNT(CASE WHEN event_type='purchase' THEN 1 END) AS total_orders,
  AVG(amount) AS avg_order_value
FROM retail_catalog.gold.fct_events
WHERE event_type='purchase'
GROUP BY category, event_date
""").write.mode("overwrite").saveAsTable("retail_catalog.gold.gold_revenue_by_category")

In [0]:
spark.sql("""
SELECT
  product_id,
  COUNT(CASE WHEN event_type='click' THEN 1 END) AS clicks,
  COUNT(CASE WHEN event_type='cart' THEN 1 END) AS carts,
  COUNT(CASE WHEN event_type='purchase' THEN 1 END) AS purchases
FROM retail_catalog.gold.fct_events
GROUP BY product_id
""").write.mode("overwrite").saveAsTable("retail_catalog.gold.gold_top_products")

In [0]:
spark.sql("""
SELECT
  user_id,
  COUNT(DISTINCT event_date) AS active_days,
  COUNT(*) AS total_events,
  SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchases
FROM retail_catalog.gold.fct_events
GROUP BY user_id
""").write.mode("overwrite").saveAsTable("retail_catalog.gold.gold_user_segments")

In [0]:
spark.sql("""
SELECT
  region_key,
  COUNT(*) AS total_events,
  SUM(CASE WHEN event_type='purchase' THEN amount ELSE 0 END) AS revenue,
  COUNT(CASE WHEN event_type='purchase' THEN 1 END) * 1.0 /
  COUNT(CASE WHEN event_type='search' THEN 1 END) AS conversion_rate
FROM retail_catalog.gold.fct_events
GROUP BY region_key
""").write.mode("overwrite").saveAsTable("retail_catalog.gold.gold_region_performance")

In [0]:
spark.sql("""
SELECT
  event_hour,
  COUNT(*) AS total_events,
  SUM(CASE WHEN event_type='purchase' THEN amount ELSE 0 END) AS revenue
FROM retail_catalog.gold.fct_events
GROUP BY event_hour
""").write.mode("overwrite").saveAsTable("retail_catalog.gold.gold_hourly_traffic")